In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import shutil

import numpy as np
import pandas as pd
import tqdm
import xarray as xr

# Functions for downloading and transforming data
import opensense_data_downloader_and_transformer as oddt

In [2]:
# Folder for downloading data
local_path = "./andersson_2022_OpenMRG/"

# Create folder for storing updated files
os.makedirs("./OpenMRG", exist_ok=True)

# Copy Torslanda event
os.makedirs("./OpenMRG/Torslanda-event-2015-07-28", exist_ok=True)

shutil.copyfile(
    "./andersson_2022_OpenMRG/Torslanda-event-2015-07-28/Torslanda-interactive-time-series.zip", 
    "./OpenMRG/Torslanda-event-2015-07-28/Torslanda-interactive-time-series.zip",
)

shutil.copyfile(
    "./andersson_2022_OpenMRG/Torslanda-event-2015-07-28/Torslanda-radar-animation.mp4", 
    "./OpenMRG/Torslanda-event-2015-07-28/Torslanda-radar-animation.mp4",
)

'./OpenMRG/Torslanda-event-2015-07-28/Torslanda-radar-animation.mp4'

# Get original OpenMRG data
source: https://zenodo.org/record/6673751

In [3]:
oddt.download_andersson_2022_OpenMRG(local_path=local_path, print_output=True)

File already exists at desired location ./andersson_2022_OpenMRG/OpenMRG.zip
Not downloading!


# Transform to opensense naming conventions

## CML data

In [4]:
# Transform first part of the data
ds1 = oddt.transform_andersson_2022_OpenMRG(
    fn=local_path + "OpenMRG.zip",  # navigate to your local sandbox clone
    path_to_extract_to=local_path,
    time_start_end=(
        None,
        "2015-07-15T00:00",
    ),  # default (None, None) -> no timeslicing. ie. ('2015-08-31T00', None),
    restructure_data=True,
)

/home/erlend/git/opensense_example_data/OpenMRG/notebooks/opensense_data_downloader_and_transformer.py:303: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'sublink' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  ds_multindex = ds.assign_coords({'sublink':df_metadata.index})


In [5]:
# Transform second part of the data
ds2 = oddt.transform_andersson_2022_OpenMRG(
    fn=local_path + "OpenMRG.zip",  # navigate to your local sandbox clone
    path_to_extract_to=local_path,
    time_start_end=(
        "2015-07-15T00:00",
        None,
    ),  # default (None, None) -> no timeslicing. ie. ('2015-08-31T00', None),
    restructure_data=True,
)

/home/erlend/git/opensense_example_data/OpenMRG/notebooks/opensense_data_downloader_and_transformer.py:303: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'sublink' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  ds_multindex = ds.assign_coords({'sublink':df_metadata.index})


In [6]:
# concat and drop overlaying duplicate
ds_cml = xr.concat([ds1, ds2], dim="time").drop_duplicates(dim="time")

In [7]:
ds_cml.attrs["file author(s)"] = "Maximilian Graf, Erlend Øydvin and Christian Chwala"
ds_cml.attrs["title"] = "Transformed and resampled OpenMRG-CML"
ds_cml.attrs["comment"] += (
    "\n\nTransformed using opensense_data_downloader_and_transformer \n"
)
ds_cml.attrs["contact"] += ", erlend.oydvin@nmbu.no"

In [8]:
# Create cml folder
os.makedirs("./OpenMRG/cml", exist_ok=True)

# Store transformed CML data
ds_cml.to_netcdf("./OpenMRG/cml/cml.nc")

## Radar data

In [9]:
# Read radar data and apply opensense naming conventions
ds_rad = (
    xr.open_dataset(local_path + "radar/radar.nc")
    .transpose("time", "y", "x")
)

# Move variables to coordinates
ds_rad.coords['lat'] = ds_rad.lat
ds_rad.coords['lon'] = ds_rad.lon
ds_rad.coords['crs'] = ds_rad.crs

In [10]:
# Make radar directory
os.makedirs("./OpenMRG/radar", exist_ok=True)

# Store transformed radar data
ds_rad.to_netcdf("./OpenMRG/radar/radar.nc")

## Gauge data

In [11]:
# Read city gauge data from CSV, apply opensense naming conventions and store to xarray
df_gauge = pd.read_csv(
    local_path + "gauges/city/CityGauges-2015JJA.csv", index_col=0, parse_dates=True
)
df_gauge_meta = pd.read_csv(local_path + "gauges/city/CityGauges-metadata.csv")

df_gauge.index = df_gauge.index.tz_localize(None).astype("datetime64[ns]")

ds_gauges = xr.Dataset(
    data_vars={"rainfall_amount": (["station_id", "time"], df_gauge.T)},
    coords={
        "id": df_gauge_meta.index.to_numpy(),
        "time": df_gauge.index.to_numpy(),
        "lon": (["station_id"], df_gauge_meta.Longitude_DecDeg),
        "lat": (["station_id"], df_gauge_meta.Latitude_DecDeg),
        "location": (["station_id"], df_gauge_meta.Location),
        "type": (["station_id"], df_gauge_meta.Type),
        "quantization": (["station_id"], df_gauge_meta["Resolution (mm)"]),
    },
)

In [12]:
# Read smhio gauge data from CSV, apply opensense naming conventions and store to xarray
df_gauge_smhi = pd.read_csv(
    local_path + "gauges/smhi/GbgA-71420-2015JJA.csv",
    index_col=0,
    parse_dates=True,
)

# Convert to no timezone to make to_numpy work instead of .values (RUFF complains)
df_gauge_smhi.index = df_gauge_smhi.index.tz_localize(None).astype("datetime64[ns]")


ds_gauges_smhi = xr.Dataset(
    data_vars={
        "rainfall_amount": (["station_id", "time"], [df_gauge_smhi.Pvol_mm.to_numpy()]),
    },
    coords={
        "id": ["SMHI"],
        "time": df_gauge_smhi.index.to_numpy(),
        "lon": (["station_id"], [11.9924]),
        "lat": (["station_id"], [57.7156]),
        "location": (["station_id"], ["Goeteburg A"]),
        "type": (["station_id"], ["15 min rainfall sum"]),
        "quantization": (["station_id"], [0.1]),
    },
)

In [13]:
# Make gauge directories
os.makedirs("./OpenMRG/gauges", exist_ok=True)
os.makedirs("./OpenMRG/gauges/smhi", exist_ok=True)
os.makedirs("./OpenMRG/gauges/city", exist_ok=True)

# Store data
ds_gauges_smhi.to_netcdf("./OpenMRG/gauges/smhi/smhi_gauge.nc")
ds_gauges.to_netcdf("./OpenMRG/gauges/city/municp_gauge.nc")

In [14]:
ds_gauges

<xarray.Dataset> Size: 12MB
Dimensions:          (station_id: 10, time: 132480, id: 10)
Coordinates:
  * time             (time) datetime64[ns] 1MB 2015-06-01T00:01:00 ... 2015-0...
  * id               (id) int64 80B 0 1 2 3 4 5 6 7 8 9
    lon              (station_id) float64 80B 11.94 12.04 12.07 ... 11.97 11.94
    lat              (station_id) float64 80B 57.65 57.72 57.75 ... 57.71 57.63
    location         (station_id) object 80B 'Järnbrottsmotet' ... 'Askim Ögä...
    type             (station_id) object 80B 'Weighing' ... 'Tipping-bucket'
    quantization     (station_id) float64 80B 0.1 0.1 0.1 0.1 ... 0.2 0.2 0.2
Dimensions without coordinates: station_id
Data variables:
    rainfall_amount  (station_id, time) float64 11MB 0.1 0.0 0.1 ... 0.2 0.0 0.0